## Using gradio to build the UI

In [16]:
import gradio as gr
# def greet(name):
#     return f"Hello, {name}"
# demo = gr.Interface(fn=greet,inputs="text", outputs="text")
# demo.launch()

In [14]:
import os 
from dotenv import load_dotenv
from IPython.display import Markdown, display
load_dotenv(override=True)
gemini_api_key = os.getenv("GOOGLE_API_KEY")
cerebras_api_key = os.getenv("CEREBRAS_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
from scraper import fetch_website_contents

In [15]:
from litellm import completion

In [19]:
system_prompt = f"""You are an expert marketing strategist, brand designer, copywriter, and brochure creation specialist.

Your task is to transform a website summary into a polished, professional, visually structured brochure suitable for business presentation, client acquisition, investor communication, or marketing distribution.

Requirements:

1. Analyze the provided website summary and identify:
   - Company name
   - Industry
   - Products and services
   - Unique value proposition
   - Key benefits
   - Target audience
   - Competitive differentiators
   - Brand positioning

2. Create brochure content that is:
   - Professional and persuasive
   - Clear and concise
   - Marketing focused
   - Factually grounded in the provided summary
   - Free from unsupported claims

3. Structure the brochure using the following sections whenever information is available:

   - Cover Page
     - Company Name
     - Tagline
     - Hero Statement

   - About Us

   - Our Services / Products

   - Key Features

   - Why Choose Us

   - Benefits for Customers

   - Industries Served

   - Customer Value Proposition

   - Call to Action

   - Contact Information (only if provided)

4. Improve readability by:
   - Using compelling headlines
   - Using concise paragraphs
   - Including bullet points where appropriate
   - Maintaining a premium business tone

5. If information is missing:
   - Do not invent facts
   - Use neutral placeholders such as:
     [Contact Information Not Provided]

6. Output Format:

   # Brochure Title

   ## Cover Section

   ## About Us

   ## Services

   ## Features

   ## Why Choose Us

   ## Benefits

   ## Call to Action

7. Additionally provide:
   - Suggested brochure design theme
   - Suggested color palette
   - Suggested imagery style
   - Recommended brochure layout (bi fold, tri fold, A4, etc.)

Return only the completed brochure content and design recommendations."""

In [17]:
def gen_user_prompt(website):
    user_prompt= f"""Create a professional brochure using the website summary provided at the end of this prompt.

    Requirements:

    - Create brochure ready marketing content.
    - Maintain a professional and modern tone.
    - Highlight the company's core value proposition.
    - Emphasize customer benefits and business impact.
    - Use compelling headlines and concise content.
    - Do not include information that is not present in the summary.
    - If any information is missing, omit the section instead of making assumptions.

    Output Structure:

    # Brochure Title

    ## Cover Section
    - Headline
    - Tagline
    - Hero Description

    ## About Us

    ## Products and Services

    ## Key Features

    ## Benefits

    ## Why Choose Us

    ## Call To Action

    ## Design Recommendations
    - Theme
    - Color Palette
    - Imagery Style
    - Typography Style
    - Layout Recommendation

    Website Summary:"""
    content = fetch_website_contents(website)
    user_prompt+= "/n".join(content)
    return user_prompt

In [23]:
def mod_call(website_link):
    stream = completion(
        model="gemini/gemini-2.5-flash",
        api_key=gemini_api_key,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": gen_user_prompt(website_link)},
        ],
        stream=True,
    )

    result = ""

    for chunk in stream:
        delta = chunk.choices[0].delta.content

        if delta:
            result += delta
            yield result

In [24]:
msg_input = gr.Text(
    label="Enter the company url",
    info="Enter the company for which brochure to be generated",
    lines=2,
)

msg_out = gr.Markdown(label="Response")

view = gr.Interface(
    fn=mod_call,
    title="Brochure Generator",
    inputs=msg_input,
    outputs=msg_out,
    flagging_mode="never",
    examples=["https://sushantazad.vercel.app/", "https://www.galaxyweblinks.com/"],
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


c:\Users\Sushant\Documents\IntelligenceStack\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
